# Expense Audit
Clean the raw expense CSV, flag problems, and summarize spend by category.

In [ ]:
import csv
from collections import Counter

In [ ]:
INPUT = "input/expenses_raw.csv"
CLEAN = "expenses_clean.csv"
FLAGGED = "expenses_flagged.csv"
FIELDS = ["date", "vendor", "amount", "category"]

In [ ]:
with open(INPUT, newline="") as f:
    rows = list(csv.DictReader(f))
print(f"Rows read: {len(rows)}")
rows[:5]

In [ ]:
for row in rows:
    row["vendor"] = row["vendor"].strip()
    row["category"] = row["category"].strip().title()
    row["date"] = row["date"].strip()
sorted({r["category"] for r in rows})

In [ ]:
seen = set()
clean, flagged, duplicates = [], [], 0
for row in rows:
    key = (row["date"], row["vendor"], row["amount"], row["category"])
    if key in seen:
        duplicates += 1
        continue
    seen.add(key)
    (clean if row["date"] else flagged).append(row)
print(f"Duplicates removed: {duplicates}")
print(f"Rows flagged (missing date): {len(flagged)}")
print(f"Rows kept: {len(clean)}")

In [ ]:
for path, data in ((CLEAN, clean), (FLAGGED, flagged)):
    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDS)
        writer.writeheader()
        writer.writerows(data)
print("Wrote", CLEAN, "and", FLAGGED)

In [ ]:
totals = Counter()
for row in clean:
    totals[row["category"]] += float(row["amount"])
for category, total in totals.most_common():
    print(f"{category}: {total:.2f}")